# Stochastic Calculus, Step by Step · Lesson 34: Numerical methods: roots, interpolation, integration, fitting

Companion notebook for the lesson. Every number in the lesson's text, tables and figures is produced by running this notebook and read from `build/outputs.json`.

> **The course's numbers.** A share priced at S₀ = 100, a strike K = 105, a risk-free rate r = 3% (continuously compounded), a volatility σ = 20% and a horizon T = 1 year. Where a lesson needs the share's real-world expected return, it is μ = 8%. Made up for teaching; every random number is simulated with a fixed seed.
>
> Educational material only, not financial advice.

1. Setup · 2. Root finding: implied volatility · 3. Interpolation: forward rates from a few zero rates · 4. Integration: Gaussian quadrature · 5. Fitting: least squares and Levenberg-Marquardt · 6. Exercises · 7. Export

**Running in Google Colab?** Nothing to install: this notebook only needs numpy, pandas and matplotlib, which Colab already has. Run the cells in order.

In [ ]:
# Parameters (papermill overrides these)
output_json = "build/outputs.json"
seed = 2026

## 1. Setup

In [ ]:
import json
import math
import datetime as dt
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

S0, K, r, sigma, T = 100.0, 105.0, 0.03, 0.20, 1.0     # the course's numbers
N = lambda x: 0.5 * (1 + math.erf(x / math.sqrt(2)))   # standard normal distribution function
out = {"meta": {
    "series": "Stochastic Calculus, Step by Step",
    "lesson": 34,
    "quantlib_version": "not used",
    "quotes_label": "simulated (no market data)",
    "seed": seed,
    "generated_at": dt.datetime.now().isoformat(timespec="seconds"),
}, "inputs": {"S0": S0, "K": K, "r": r, "sigma": sigma, "T": T, "r_pct": r * 100, "sigma_pct": sigma * 100}}

In [ ]:
mu = 0.08                         # the share's real-world expected return
theta_mpr = (mu - r) / sigma      # the market price of risk
Nv = np.vectorize(N)              # N, for arrays
phi = lambda x: np.exp(-np.asarray(x, float) ** 2 / 2) / math.sqrt(2 * math.pi)   # the normal density


def d_plus(s, tau, sig=sigma, k=K):
    return (np.log(s / k) + (r + 0.5 * sig**2) * tau) / (sig * np.sqrt(tau))


def d_minus(s, tau, sig=sigma, k=K):
    return d_plus(s, tau, sig, k) - sig * np.sqrt(tau)


def call(s, tau, sig=sigma, k=K):
    """The Black-Scholes call price: Lesson 20 derives it."""
    return s * Nv(d_plus(s, tau, sig, k)) - k * np.exp(-r * tau) * Nv(d_minus(s, tau, sig, k))


def put(s, tau, sig=sigma, k=K):
    return call(s, tau, sig, k) - s + k * np.exp(-r * tau)


out["bs"] = {"mu": mu, "theta": theta_mpr, "d_plus": float(d_plus(S0, T)), "d_minus": float(d_minus(S0, T)),
             "N_plus": N(float(d_plus(S0, T))), "N_minus": N(float(d_minus(S0, T))),
             "call": float(call(S0, T)), "put": float(put(S0, T))}
out["bs"]

## 2. Root finding: implied volatility

The call is worth 7.1281 at σ = 20%. Given only the price, find σ: solve $C(\sigma) - 7.1281 = 0$. **Bisection** halves a bracket; **Newton** follows the tangent, using vega (Lesson 21) as the slope; **Brent** mixes a bracket with interpolation steps. Errors in σ after each iteration:

In [ ]:
target = float(call(S0, T))
vega = lambda sig, k=K: float(S0 * phi(d_plus(S0, T, sig, k)) * math.sqrt(T))
f_ = lambda sig, k=K, c=target: float(call(S0, T, sig, k)) - c


def bisection(f, a, b, tol=1e-10, maxit=60):
    fa = f(a); xs = []
    for _ in range(maxit):
        m = 0.5 * (a + b); fm = f(m); xs.append(m)
        if (fm > 0) == (fa > 0):
            a, fa = m, fm
        else:
            b = m
        if b - a < tol:
            break
    return xs


def newton(f, df, x0, n=8):
    xs = [x0]
    for _ in range(n):
        x = xs[-1]
        d = df(x)
        xs.append(x - f(x) / d if d > 0 else float("nan"))
        if not math.isfinite(xs[-1]) or abs(xs[-1] - x) < 1e-15:
            break
    return xs


def brent(f, a, b, tol=1e-12, maxit=60):
    # Brent's method (as in Numerical Recipes): bisection, secant and inverse quadratic interpolation, safely bracketed.
    fa, fb = f(a), f(b)
    c, fc = b, fb
    d = e = b - a
    xs = []
    for _ in range(maxit):
        if (fb > 0) == (fc > 0):
            c, fc = a, fa
            d = e = b - a
        if abs(fc) < abs(fb):
            a, b, c = b, c, b
            fa, fb, fc = fb, fc, fb
        tol1 = 2e-16 * abs(b) + 0.5 * tol
        xm = 0.5 * (c - b)
        if abs(xm) <= tol1 or fb == 0:
            break
        if abs(e) >= tol1 and abs(fa) > abs(fb):
            s_ = fb / fa
            if a == c:
                p_, q_ = 2 * xm * s_, 1 - s_
            else:
                q_, rr_ = fa / fc, fb / fc
                p_ = s_ * (2 * xm * q_ * (q_ - rr_) - (b - a) * (rr_ - 1))
                q_ = (q_ - 1) * (rr_ - 1) * (s_ - 1)
            if p_ > 0:
                q_ = -q_
            p_ = abs(p_)
            if 2 * p_ < min(3 * xm * q_ - abs(tol1 * q_), abs(e * q_)):
                e, d = d, p_ / q_
            else:
                d = e = xm
        else:
            d = e = xm
        a, fa = b, fb
        b += d if abs(d) > tol1 else math.copysign(tol1, xm)
        fb = f(b)
        xs.append(b)
    return xs


bis = bisection(f_, 0.01, 1.0)
new = newton(f_, vega, 0.5)
bre = brent(f_, 0.01, 1.0)
err = lambda xs: [abs(x - 0.2) for x in xs]
root_rows = [{"it": i + 1, "bisection": (err(bis) + [None] * 40)[i], "newton": (err(new[1:]) + [None] * 40)[i], "brent": (err(bre) + [None] * 40)[i]}
             for i in range(4)]
# deep out of the money: K = 200, true sigma 20%, Newton started at 5%
k_far = 200.0
c_far = float(call(S0, T, 0.2, k_far))
new_far = newton(lambda s_: f_(s_, k_far, c_far), lambda s_: vega(s_, k_far), 0.05, n=3)
bre_far = brent(lambda s_: f_(s_, k_far, c_far), 0.01, 1.0)
out["roots"] = {"target": target, "rows": root_rows, "n_bis": len(bis), "n_new": len(new) - 1, "n_bre": len(bre),
                "newton_path": new[:5], "far_price": c_far, "far_vega0": vega(0.05, k_far), "far_newton": new_far[:3],
                "far_brent": bre_far[-1], "far_brent_n": len(bre_far),
                "fig": {"bis": err(bis), "new": err(new[1:]), "bre": err(bre)}}
pd.DataFrame(root_rows)

## 3. Interpolation: forward rates from a few zero rates

Made-up continuously compounded zero rates at 1, 2, 3, 5 and 10 years: 3.0%, 3.3%, 3.5%, 3.8% and 4.1%. Fill the gaps three ways: straight lines, a **natural cubic spline** (solved with Lesson 29's tridiagonal method) and a **monotone cubic** that can't overshoot the data. Judge them by the **forward rates** $f(t) = z(t) + t\,z'(t)$ they imply. A second made-up curve, with a step from 3% to 4%, shows what "overshoot" means.

In [ ]:
tp = np.array([1.0, 2.0, 3.0, 5.0, 10.0])
zp = np.array([3.0, 3.3, 3.5, 3.8, 4.1]) / 100


def thomas(a, b, c, d):
    # Solve a tridiagonal system: a sub-diagonal, b diagonal, c super-diagonal (Lesson 29).
    n = len(b); c_ = np.zeros(n); d_ = np.zeros(n)
    c_[0], d_[0] = c[0] / b[0], d[0] / b[0]
    for i in range(1, n):
        m = b[i] - a[i] * c_[i - 1]
        c_[i] = c[i] / m if i < n - 1 else 0.0
        d_[i] = (d[i] - a[i] * d_[i - 1]) / m
    x = np.zeros(n); x[-1] = d_[-1]
    for i in range(n - 2, -1, -1):
        x[i] = d_[i] - c_[i] * x[i + 1]
    return x


def natural_spline(x, y):
    h = np.diff(x); n = len(x)
    a = np.zeros(n); b = np.ones(n); c = np.zeros(n); d = np.zeros(n)
    for i in range(1, n - 1):
        a[i], b[i], c[i] = h[i - 1], 2 * (h[i - 1] + h[i]), h[i]
        d[i] = 6 * ((y[i + 1] - y[i]) / h[i] - (y[i] - y[i - 1]) / h[i - 1])
    M = thomas(a, b, c, d)           # second derivatives at the knots; zero at both ends

    def ev(t):
        i = np.clip(np.searchsorted(x, t) - 1, 0, n - 2)
        hh = h[i]; u = (x[i + 1] - t); v = (t - x[i])
        val = M[i] * u**3 / (6 * hh) + M[i + 1] * v**3 / (6 * hh) + (y[i] / hh - M[i] * hh / 6) * u + (y[i + 1] / hh - M[i + 1] * hh / 6) * v
        der = -M[i] * u**2 / (2 * hh) + M[i + 1] * v**2 / (2 * hh) - (y[i] / hh - M[i] * hh / 6) + (y[i + 1] / hh - M[i + 1] * hh / 6)
        return val, der
    return ev, M


def monotone_cubic(x, y):
    # Fritsch-Carlson: Hermite cubic with slopes limited so the curve never overshoots the data.
    h = np.diff(x); dl = np.diff(y) / h; n = len(x)
    m = np.zeros(n); m[0], m[-1] = dl[0], dl[-1]
    for i in range(1, n - 1):
        if dl[i - 1] * dl[i] > 0:
            w1, w2 = 2 * h[i] + h[i - 1], h[i] + 2 * h[i - 1]
            m[i] = (w1 + w2) / (w1 / dl[i - 1] + w2 / dl[i])
    def ev(t):
        i = np.clip(np.searchsorted(x, t) - 1, 0, n - 2)
        hh = h[i]; s = (t - x[i]) / hh
        h00, h10, h01, h11 = 2*s**3 - 3*s**2 + 1, s**3 - 2*s**2 + s, -2*s**3 + 3*s**2, s**3 - s**2
        val = h00 * y[i] + h10 * hh * m[i] + h01 * y[i + 1] + h11 * hh * m[i + 1]
        d00, d10, d01, d11 = 6*s**2 - 6*s, 3*s**2 - 4*s + 1, -6*s**2 + 6*s, 3*s**2 - 2*s
        der = (d00 * y[i] + d01 * y[i + 1]) / hh + d10 * m[i] + d11 * m[i + 1]
        return val, der
    return ev


tt = np.linspace(1.0, 10.0, 451)
lin_z = np.interp(tt, tp, zp)
seg = np.clip(np.searchsorted(tp, tt, side="right") - 1, 0, len(tp) - 2)
lin_dz = (np.diff(zp) / np.diff(tp))[seg]
spl, M_knots = natural_spline(tp, zp)
mon = monotone_cubic(tp, zp)
zs_, dzs_ = spl(tt); zm_, dzm_ = mon(tt)
fwd = {"linear": lin_z + tt * lin_dz, "spline": zs_ + tt * dzs_, "monotone": zm_ + tt * dzm_}
interp_rows = [{"method": k, "fwd_min": float(v.min() * 100), "fwd_max": float(v.max() * 100), "fwd_10y": float(v[-1] * 100),
                "z_7y": float({"linear": lin_z, "spline": zs_, "monotone": zm_}[k][np.argmin(abs(tt - 7))] * 100)} for k, v in fwd.items()]
def turning(v):
    # local maxima and minima of a sampled curve, as (t, value in %)
    d = np.sign(np.diff(v))
    idx = [i + 1 for i in range(len(d) - 1) if d[i] != 0 and d[i + 1] != 0 and d[i] != d[i + 1]]
    return [{"t": float(tt[i]), "f": float(v[i] * 100)} for i in idx]


for row in interp_rows:
    row["turns"] = len(turning(fwd[row["method"]]))
# a second made-up curve with a step: 3% out to 3 years, 4% from 5 years
zq = np.array([3.0, 3.0, 3.0, 4.0, 4.0]) / 100
spl_q, _ = natural_spline(tp, zq)
zs_q = spl_q(tt)[0]; zm_q = monotone_cubic(tp, zq)(tt)[0]
out["interp"] = {"t": tp.tolist(), "z_pct": (zp * 100).tolist(), "rows": interp_rows, "M": M_knots.tolist(),
                 "turns_spline": turning(fwd["spline"]), "turns_monotone": turning(fwd["monotone"]),
                 "step": {"spline_min": float(zs_q.min() * 100), "spline_max": float(zs_q.max() * 100),
                          "mono_min": float(zm_q.min() * 100), "mono_max": float(zm_q.max() * 100)},
                 "fig2": {"spline": (zs_q * 100).round(4).tolist(), "monotone": (zm_q * 100).round(4).tolist(), "z_pct": (zq * 100).tolist()},
                 "lin_jump_2_lo": float((zp[1] + 2 * (zp[1] - zp[0])) * 100), "lin_jump_2_hi": float((zp[1] + 2 * (zp[2] - zp[1]) / 1) * 100),
                 "fig": {"t": tt.round(3).tolist(), **{k: (v * 100).round(4).tolist() for k, v in fwd.items()}}}
pd.DataFrame(interp_rows).round(3)

## 4. Integration: Gaussian quadrature

The call as an integral over the standard normal, $e^{-rT}\int\max(S_T(z) - K, 0)\,\varphi(z)\,dz$. The trapezium rule (Lesson 20) needs many points. **Gaussian quadrature** chooses where to put a few points so that polynomials are integrated exactly: Gauss-Hermite for an integral against the normal density over the whole line, Gauss-Legendre on an interval. The payoff's kink at $z^* = -d_-$ spoils Gauss-Hermite; starting the interval at the kink fixes it.

In [ ]:
z_star = -float(d_minus(S0, T))
pay_z = lambda z: math.exp(-r * T) * np.maximum(S0 * np.exp((r - 0.5 * sigma**2) * T + sigma * math.sqrt(T) * z) - K, 0)
quad_rows = []
for n_ in (5, 10, 20, 40, 80):
    zt = np.linspace(-8, 8, n_); wt = np.full(n_, zt[1] - zt[0]); wt[[0, -1]] /= 2
    trap = float((pay_z(zt) * phi(zt) * wt).sum())
    xh, wh = np.polynomial.hermite.hermgauss(n_)
    gh = float((wh * pay_z(math.sqrt(2) * xh)).sum() / math.sqrt(math.pi))
    xl, wl = np.polynomial.legendre.leggauss(n_)
    lo, hi = z_star, 8.0
    zl = 0.5 * (hi - lo) * xl + 0.5 * (hi + lo)
    gl = float((0.5 * (hi - lo) * wl * pay_z(zl) * phi(zl)).sum())
    quad_rows.append({"n": n_, "trap": trap - target, "hermite": gh - target, "legendre": gl - target})
out["quad"] = {"z_star": z_star, "rows": quad_rows}
pd.DataFrame(quad_rows)

## 5. Fitting: least squares and Levenberg-Marquardt

**Linear least squares.** Made-up implied volatilities at seven strikes, against log-moneyness $k = \ln(K/F)$ with $F = S_0e^{rT}$. Fit a parabola $a + bk + ck^2$ by solving the normal equations $X^\top X\beta = X^\top y$.

**Nonlinear least squares.** Made-up zero rates at eight maturities, fitted with a three-parameter curve $z(t) = b_0 + b_1\frac{1 - e^{-t/\tau}}{t/\tau}$ (Nelson-Siegel without its hump term). It is nonlinear in $\tau$. **Gauss-Newton** linearises and solves; **Levenberg-Marquardt** adds a damping $\lambda$ that shortens the step when the linearisation can't be trusted. From the same poor start, compare the sum of squared errors at each iteration.

In [ ]:
F_ = S0 * math.exp(r * T)
Ks = np.array([80, 90, 95, 100, 105, 110, 120.0])
kx = np.log(Ks / F_)
noise_v = np.array([-1.2, 0.8, -0.5, 1.0, -0.7, 0.4, -0.9]) / 1000
vols = 0.20 - 0.10 * kx + 0.30 * kx**2 + noise_v
X = np.column_stack([np.ones_like(kx), kx, kx**2])
beta = np.linalg.solve(X.T @ X, X.T @ vols)
resid = vols - X @ beta

tm = np.array([1, 2, 3, 5, 7, 10, 20, 30.0])
gfun = lambda t, tau: (1 - np.exp(-t / tau)) / (t / tau)
model = lambda p: p[0] + p[1] * gfun(tm, p[2])
noise_y = np.array([1, -1, 0.5, -0.5, 1, -1, 0.5, -0.5]) * 1e-4
yld = 0.042 - 0.014 * gfun(tm, 2.0) + noise_y


def jac(p):
    x = tm / p[2]
    dg = (np.exp(-x) * (1 + x) - 1) / x**2
    return np.column_stack([np.ones_like(tm), gfun(tm, p[2]), p[1] * dg * (-tm / p[2] ** 2)])


def fit(p0, lam0, iters=25):
    p = np.array(p0, float); lam = lam0; sse = []
    for _ in range(iters):
        res = yld - model(p); cur = float(res @ res); sse.append(cur)
        J = jac(p); A = J.T @ J; g_ = J.T @ res
        if lam == 0:                       # Gauss-Newton: take the full step
            p = p + np.linalg.solve(A, g_)
            continue
        while lam < 1e12:                  # Levenberg-Marquardt: accept only steps that lower the error
            step = np.linalg.solve(A + lam * np.diag(np.diag(A)), g_)
            pn = p + step
            if pn[2] > 0 and float((yld - model(pn)) @ (yld - model(pn))) < cur:
                p, lam = pn, lam / 3
                break
            lam *= 3
    return p, sse


start_p = [0.05, -0.03, 20.0]
p_gn, sse_gn = fit(start_p, 0.0)
p_lm, sse_lm = fit(start_p, 1e-2)
out["fit"] = {"F": F_, "k": kx.tolist(), "vols": vols.tolist(), "beta": beta.tolist(), "beta_true": [0.20, -0.10, 0.30],
              "resid_sd": float(resid.std(ddof=3)), "t": tm.tolist(), "y_pct": (yld * 100).tolist(), "start": start_p,
              "gn": p_gn.tolist(), "lm": p_lm.tolist(), "true": [0.042, -0.014, 2.0], "sse_gn": sse_gn, "sse_lm": sse_lm,
              "fig": {"tt": np.linspace(0.5, 30, 120).round(3).tolist(),
                      "fit_lm": (100 * (p_lm[0] + p_lm[1] * gfun(np.linspace(0.5, 30, 120), p_lm[2]))).round(4).tolist()}}
{"beta": beta.round(4).tolist(), "gauss_newton": p_gn.round(5).tolist(), "levenberg_marquardt": p_lm.round(5).tolist()}

In [ ]:
# Chart data for the video (no random numbers)
def lg(v):
    return round(math.log10(max(abs(v), 1e-16)), 4)
def padded(errs, n):
    return [lg(errs[min(i, len(errs) - 1)]) for i in range(n)]
fr = out["roots"]["fig"]
out["roots"]["fig2"] = {"it": list(range(1, 11)), "bis": padded(fr["bis"], 10), "new": padded(fr["new"], 10), "bre": padded(fr["bre"], 10),
                        "ticks": [{"x": float(i), "label": str(i)} for i in (1, 2, 4, 6, 8, 10)]}
out["fit"]["fig2"] = {"it": list(range(len(sse_gn))), "gn": [lg(v) for v in sse_gn], "lm": [lg(v) for v in sse_lm],
                      "ticks": [{"x": float(i), "label": str(i)} for i in (0, 5, 10, 15, 20)]}

## 6. Exercises

Try each one before looking anything up. Full solutions are in the companion volume, *Part 5: Solutions*.

**Exercise 1.** Bisection starts from the bracket $[0.01, 1]$. How many halvings does it need to pin $\sigma$ down to $10^{-8}$? Newton's error roughly squares each step: $e_{n+1} \approx e_n^2$. Starting from an error of $10^{-2}$, how many steps does it need to reach $10^{-12}$?

**Exercise 2.** Derive Newton's update $\sigma_{n+1} = \sigma_n - \big(C(\sigma_n) - C^*\big)/\mathcal V(\sigma_n)$ from a first-order Taylor expansion. Explain why it fails when vega is nearly zero, as it is for a deep out-of-the-money option at a low starting volatility. What does Brent's method do instead?

**Exercise 3.** A natural cubic spline through $(t_i, y_i)$ is a cubic on each interval, with matching values, slopes and second derivatives at the knots, and zero second derivative at both ends. Writing $M_i$ for the second derivative at $t_i$, derive the equation $h_{i-1}M_{i-1} + 2(h_{i-1} + h_i)M_i + h_iM_{i+1} = 6\Big(\frac{y_{i+1} - y_i}{h_i} - \frac{y_i - y_{i-1}}{h_{i-1}}\Big)$, where $h_i = t_{i+1} - t_i$.

**Exercise 4.** The forward rate is $f(t) = z(t) + t\,z'(t)$. With straight-line interpolation of the zero rates 3.0% (1 year), 3.3% (2 years) and 3.5% (3 years), find the forward rate just before and just after 2 years. Why is a jump in the forward curve a problem for pricing?

**Exercise 5.** Show that the two-point Gauss-Legendre rule $\int_{-1}^1 f(x)\,dx \approx f(-1/\sqrt3) + f(1/\sqrt3)$ is exact for $1$, $x$, $x^2$ and $x^3$, and not for $x^4$. Why can a rule with $n$ well-chosen points be exact up to degree $2n - 1$?

**Exercise 6.** Derive the normal equations $X^\top X\beta = X^\top y$ by setting the derivative of $\|y - X\beta\|^2$ to zero. Fit a straight line $y = a + bx$ by hand to the points $(0, 1)$, $(1, 2)$ and $(2, 2)$.

**Exercise 7.** (Code) Build an implied-volatility solver that uses Newton's method and falls back to Brent when a step leaves the bracket. Test it on strikes from 50 to 200 and maturities from 0.1 to 5 years (200 cases) at true volatilities between 10% and 60%. Report the largest error, and how often plain Newton from a 20% start fails.

**Exercise 8.** (Code) Fit the three-parameter curve of this lesson with Levenberg-Marquardt from the starts $(5\%, -3\%, 20)$, $(3\%, 1\%, 0.5)$ and $(4\%, 0\%, 2)$. Which runs fail, and why? Add the bound $\tau \ge 0.05$ and try again.

## 7. Export

In [ ]:
path = Path(output_json)
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(json.dumps(out, indent=2, default=float))
print("wrote", path.resolve(), f"({path.stat().st_size / 1024:.0f} KB)")